``` text
1. Load df_chemistry / hmof_linker_extracted.csv
2. Confirm linker columns: linker_1 ... linker_13
3. Elemental composition features
   - C, N, O, F, S, Cl, Br, I counts
   - total heteroatoms
   - heteroatom fraction
4. Functional group features
   - amine
   - hydroxyl
   - carboxylate/carboxylic acid
   - fluorinated groups
   - sulfonyl/sulfate if useful
5. Extra physicochemical descriptors
   - MolWt
   - TPSA
   - HBD/HBA
   - aromatic atom count
   - aromatic fraction
   - heavy atom count
   - ring count
   - rotatable bonds
6. Descriptor validation
   - missing values
   - constant columns
   - distributions
   - correlations
7. Save final dataset
   - hmof_linker_extended_descriptors.csv

In [1]:
import pandas as pd 
import numpy as np
import re

In [13]:
df = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/hmof_linker_extracted_final.csv')

# Create a list of linker columns only
# We exclude "linker_smiles" because it stores the full linker list,
# while linker_1 to linker_13 store individual linker SMILES.

linker_cols = []
for col in df.columns:
    if col.startswith("linker_") and col != "linker_smiles":
         linker_cols.append(col) # if this column starts with linker_, save it in my list.
print(linker_cols)  
print()
print(df.shape)
print()
print("Number of linker columns:", len(linker_cols)) # Confirm linker columns 
print()

# Check how many non-missing linker SMILES are present in each linker column
# This helps us understand how many MOFs have 1 linker, 2 linkers, etc.
for col in linker_cols:
    print(col,df[col].notna().sum())

['linker_1', 'linker_2', 'linker_3', 'linker_4', 'linker_5', 'linker_6', 'linker_7', 'linker_8', 'linker_9', 'linker_10', 'linker_11', 'linker_12', 'linker_13']

(25928, 27)

Number of linker columns: 13

linker_1 25928
linker_2 24899
linker_3 20414
linker_4 1250
linker_5 261
linker_6 83
linker_7 29
linker_8 15
linker_9 10
linker_10 4
linker_11 2
linker_12 2
linker_13 1


we will create MOF-level summary features,
- total_C_count
- total_N_count
- total_O_count
- total_F_count
-total_S_count
- total_heteroatom_count
- heteroatom_fraction
- num_linkers

In [24]:
# Import RDKit for reading SMILES strings as molecule objects
from rdkit import Chem
# Define a function to count a specific element in one linker SMILES
# Example: count_element(smiles, "N") counts nitrogen atoms in that linker.
def count_element(smiles, element_symbol):
    
    # If the linker value is missing, return 0
    if pd.isna(smiles):
        return 0

    # Convert the SMILES string into an RDKit molecule object
    mol = Chem.MolFromSmiles(smiles)

    # If RDKit cannot read the SMILES, return 0
    if mol is None:
        return 0

    # Start the atom count at 0
    count = 0

    # Loop through atoms in the molecule
    for atom in mol.GetAtoms():

        # If the atom symbol matches the requested element, add 1
        if atom.GetSymbol() == element_symbol:
            count += 1

    # Return the final count
    return count
    

In [25]:
# Test the function on the first MOF's first linker
test_smiles = df.loc[0, "linker_1"]

# Display the test linker SMILES
print("Test linker SMILES:", test_smiles)

# Count selected elements in the test linker
print("C count:", count_element(test_smiles, "C"))
print("N count:", count_element(test_smiles, "N"))
print("O count:", count_element(test_smiles, "O"))
print("F count:", count_element(test_smiles, "F"))
print("S count:", count_element(test_smiles, "S"))

Test linker SMILES: O=C([O-])c1ccc(C(=O)[O-])cc1
C count: 8
N count: 0
O count: 4
F count: 0
S count: 0


In [27]:
# Define the elements we want to count in all linker columns
# These are chemically relevant elements for CO2 adsorption-related linker chemistry.
elements_to_count = ["C", "N", "O", "F", "S", "Cl", "Br", "I"]
# For each element, count how many atoms of that element appear across all linker columns
# This creates one MOF-level feature per element, such as total_C_count and total_N_count.
for element in elements_to_count:
    
    # Create a temporary list to store the total count for each MOF
    total_counts = []
    
    # Loop through each row/MOF in the dataset
    for idx, row in df.iterrows():
        
        # Start total count for this MOF at 0
        mof_total = 0
        
        # Loop through all linker columns for this MOF
        for col in linker_cols:
            
            # Add the element count from this linker to the MOF total
            mof_total += count_element(row[col], element)
        
        # Save the total count for this MOF
        total_counts.append(mof_total)
    
    # Add the total element count as a new column in the dataframe
    df[f"total_{element}_count"] = total_counts

In [28]:
# Display the new elemental composition columns for the first few MOFs
element_count_cols = [f"total_{element}_count" for element in elements_to_count]

df[element_count_cols].head()

,total_C_count,total_N_count,total_O_count,total_F_count,total_S_count,total_Cl_count,total_Br_count,total_I_count
0,8,0,4,0,0,0,0,0
1,8,0,4,0,0,0,0,0
2,8,0,4,3,0,0,0,0
3,19,0,11,0,0,0,0,0
4,18,0,8,0,0,0,0,0


In [34]:
# total_heteroatom_count means:the total number of non-carbon/non-hydrogen atoms in all linkers of one MOF.
#For this project, we can define heteroatoms as:
# Define which elements we will treat as heteroatoms
# Heteroatoms are atoms other than carbon and hydrogen.
heteroatom_elements = ["N", "O", "F", "S", "Cl", "Br", "I"]
# create a list of the hetroatom count columns already created 
heteroatom_count_cols =[]
for elements in heteroatom_elements:
    heteroatom_count_cols.append(f"total_{elements}_count")
print(heteroatom_count_cols)
print()
# Calculate the total number of heteroatom acrros all linker in each MOF
df["total_heteroatom_count"] = df[heteroatom_count_cols].sum(axis=1)
print()
# check the new heteroatom feature
df[["total_heteroatom_count"]+heteroatom_count_cols].head()

['total_N_count', 'total_O_count', 'total_F_count', 'total_S_count', 'total_Cl_count', 'total_Br_count', 'total_I_count']




,total_heteroatom_count,total_N_count,total_O_count,total_F_count,total_S_count,total_Cl_count,total_Br_count,total_I_count
0,4,0,4,0,0,0,0,0
1,4,0,4,0,0,0,0,0
2,7,0,4,3,0,0,0,0
3,11,0,11,0,0,0,0,0
4,8,0,8,0,0,0,0,0


---
* `Heteroatom` means: an atom in an organic molecule that is not carbon and not hydrogen.

* `Heavy atom` means: any atom that is not hydrogen.

- heavy atoms = linker size
- heteroatoms = chemically polar/non-carbon part
- heteroatom fraction = how heteroatom-rich the linker is

**heteroatom fraction is useful because it measures how chemically polar/heteroatom-rich the linker system is, independent of linker size.**

---


In [42]:
# Define all heavy atoms we counted in the linker features
# Heavy atoms are non-hydrogen atoms.
heavy_atom_elements = ["C", "N", "O", "F", "S", "Cl", "Br", "I"]

# Create a list of the heavy atom count columns
heavy_atom_count_cols = []

for element in heavy_atom_elements:
    heavy_atom_count_cols.append(f"total_{element}_count")
print(heavy_atom_count_cols)
print()
# Calculate the total number of heavy atoms across all linkers in each MOF
df["total_heavy_atom_count"] = df[heavy_atom_count_cols].sum(axis =1)
# calculate heteroatom fraction 
# If total_heavy_atom_count is 0 , return 0 to avoid division by zero.
df["heteroatom_fraction"] = df["total_heteroatom_count"]/df["total_heavy_atom_count"]
df["heteroatom_fraction"] = df ["heteroatom_fraction"].fillna(0)

# Check the new features
df[
    [
        "total_heavy_atom_count",
        "total_heteroatom_count",
        "heteroatom_fraction"
    ]
].head()

['total_C_count', 'total_N_count', 'total_O_count', 'total_F_count', 'total_S_count', 'total_Cl_count', 'total_Br_count', 'total_I_count']



,total_heavy_atom_count,total_heteroatom_count,heteroatom_fraction
0,12,4,0.333333
1,12,4,0.333333
2,15,7,0.466667
3,30,11,0.366667
4,26,8,0.307692


----
``` text
4. Functional group features
   - amine
   - hydroxyl
   - carboxylate/carboxylic acid
   - fluorinated groups
   - sulfonyl/sulfate if useful
```

| Feature type | Question it answers |
|---|---|
| Element count | “How many N/O/F/S atoms are present?” |
| Functional group | “What chemical environment are those atoms in?” |

``` text
Functional Group Features

In this section, SMARTS patterns are used to identify chemically meaningful
functional groups in the extracted linker SMILES. These features summarize
whether linkers contain groups that may influence CO2 adsorption through
polarity, hydrogen bonding, or electrostatic interactions.

we will use SMARTS patterns for common chemical functional groups provide precise structural definitions for substructure searching in molecular databases
```
----


In [53]:
# SMARTS is a chemical pattern language used by RDKit to search substructures.
functional_group_smarts = {
    "amine": "[NX3;H2,H1,H0;!$(NC=O)]",
    "hydroxyl": "[OX2H]",
    "carboxylate_or_carboxylic_acid": "[CX3](=O)[O-,OH]",
    "fluorinated_group": "[F]",
    "sulfonyl_or_sulfate": "[SX4](=O)(=O)"}

# Convert each SMART pattern  into an RDKit molecule patteren
# RDKit needs these patteren objects before it can search inside molecules
functional_group_patterns = {}
for group_name, smarts in functional_group_smarts.items():
    functional_group_patterns[group_name] =Chem.MolFromSmarts(smarts)

# Define a function to count one functional group pattern in one linker SMILES
def count_functional_group(smiles, pattern):
    
    # If the linker value is missing, return 0
    if pd.isna(smiles):
        return 0
    
    # Convert the linker SMILES into an RDKit molecule
    mol = Chem.MolFromSmiles(smiles)
    
    # If RDKit cannot read the SMILES, return 0
    if mol is None:
        return 0
    
    # Find all matches to the SMARTS pattern
    matches = mol.GetSubstructMatches(pattern)
    
    # Return the number of matches
    return len(matches)

In [54]:
# Test functional group counting on the first linker
test_smiles = df.loc[0, "linker_1"]

print("Test linker SMILES:", test_smiles)

for group_name, pattern in functional_group_patterns.items():
    print(group_name, count_functional_group(test_smiles, pattern))

Test linker SMILES: O=C([O-])c1ccc(C(=O)[O-])cc1
amine 0
hydroxyl 0
carboxylate_or_carboxylic_acid 2
fluorinated_group 0
sulfonyl_or_sulfate 0


In [56]:
# For each functional group, count how many times it appears across all linkers in each MOF
# This creates MOF-level features such as total_amine_count and total_hydroxyl_count.
for group_name, pattern in functional_group_patterns.items():
    
    # Create a temporary list to store the total group count for each MOF
    total_group_counts = []
    
    # Loop through each row/MOF in the dataset
    for idx, row in df.iterrows():
        
        # Start the functional group count for this MOF at 0
        mof_group_total = 0
        
        # Loop through all linker columns for this MOF
        for col in linker_cols:
            
            # Add the functional group count from this linker to the MOF total
            mof_group_total += count_functional_group(row[col], pattern)
        
        # Save the total functional group count for this MOF
        total_group_counts.append(mof_group_total)
    
    # Add the total functional group count as a new column in the dataframe
    df[f"total_{group_name}_count"] = total_group_counts

In [57]:
# Create a list of the new functional group feature columns
functional_group_cols = []

for group_name in functional_group_patterns.keys():
    functional_group_cols.append(f"total_{group_name}_count")

# Display the first few rows of functional group features
df[functional_group_cols].head()

,total_amine_count,total_hydroxyl_count,total_carboxylate_or_carboxylic_acid_count,total_fluorinated_group_count,total_sulfonyl_or_sulfate_count
0,0,0,2,0,0
1,0,0,2,0,0
2,0,0,2,3,0
3,0,0,4,0,0
4,0,0,4,0,0


---

Define and document the substructure rules used to identify functional groups.
Identify relevant groups, including amines, hydroxyls, fluorinated groups, and carboxylate-related groups.
Summarize group counts or presence across each MOF’s linker.

---


In [58]:
# Create a list of the new functional group feature columns
functional_group_cols = []

for group_name in functional_group_patterns.keys():
    functional_group_cols.append(f"total_{group_name}_count")

# Display the first few rows of functional group features
df[functional_group_cols].head()

,total_amine_count,total_hydroxyl_count,total_carboxylate_or_carboxylic_acid_count,total_fluorinated_group_count,total_sulfonyl_or_sulfate_count
0,0,0,2,0,0
1,0,0,2,0,0
2,0,0,2,3,0
3,0,0,4,0,0
4,0,0,4,0,0


``` text
5. Extra physicochemical descriptors
total_MolWt
mean_MolWt
max_MolWt
total_TPSA
mean_TPSA
total_HBD
total_HBA
total_aromatic_atom_count
aromatic_fraction
total_heavy_atom_count
total_ring_count
total_rotatable_bond_count
6. Descriptor validation
   - missing values
   - constant columns
   - distributions
   - correlations

In [71]:
# Import RDkit descriptor modules 
from rdkit.Chem import Descriptors 
from rdkit.Chem import rdMolDescriptors, Lipinski, rdMolDescriptors
# Define a function to calculate physicochemical descriptors for one linker SMILES
def calculate_linker_descriptors(smiles):
    
    # If the linker value is missing, return None
    if pd.isna(smiles):
        return None
    
    # Convert SMILES into an RDKit molecule
    mol = Chem.MolFromSmiles(smiles)
    
    # If RDKit cannot read the SMILES, return None
    if mol is None:
        return None
    
    # Count aromatic atoms
    aromatic_atom_count = 0
    
    for atom in mol.GetAtoms():
        if atom.GetIsAromatic():
            aromatic_atom_count += 1
    
    # Count heavy atoms
    heavy_atom_count = mol.GetNumHeavyAtoms()
    
    # Calculate aromatic fraction
    # This is the fraction of heavy atoms that are aromatic.
    if heavy_atom_count > 0:
        aromatic_fraction = aromatic_atom_count / heavy_atom_count
    else:
        aromatic_fraction = 0
    
    # Store descriptor values in a dictionary
    descriptors = {
        "MolWt": Descriptors.MolWt(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        "aromatic_atom_count": aromatic_atom_count,
        "aromatic_fraction": aromatic_fraction,
        "heavy_atom_count": heavy_atom_count,
        "ring_count": rdMolDescriptors.CalcNumRings(mol),
        "rotatable_bond_count": rdMolDescriptors.CalcNumRotatableBonds(mol)
    }
    
    return descriptors


In [72]:
# Test descriptor calculation on the first linker
test_smiles = df.loc[0, "linker_1"]

test_descriptors = calculate_linker_descriptors(test_smiles)

test_descriptors

{'MolWt': 164.11599999999999,
 'TPSA': 80.25999999999999,
 'HBD': 0,
 'HBA': 4,
 'aromatic_atom_count': 6,
 'aromatic_fraction': 0.5,
 'heavy_atom_count': 12,
 'ring_count': 1,
 'rotatable_bond_count': 2}

In [ ]:
# Create empty lists for MOF-level descriptor summaries
total_MolWt = []
mean_MolWt = []
max_MolWt = []

total_TPSA = []
mean_TPSA = []
max_TPSA = []

total_HBD = []
total_HBA = []

total_aromatic_atom_count = []
mean_aromatic_fraction = []

total_heavy_atom_count_rdkit = []
total_ring_count = []
total_rotatable_bond_count = []

# Loop through each MOF row
for idx, row in df.iterrows():
    
    # Store descriptor dictionaries for valid linkers in this MOF
    linker_descs = []
    
    # Loop through linker_1 to linker_13
    for col in linker_cols:
        desc = calculate_linker_descriptors(row[col])
        
        if desc is not None:
            linker_descs.append(desc)
    
    # Summarize descriptors if the MOF has valid linkers
    if len(linker_descs) > 0:
        molwt_values = [d["MolWt"] for d in linker_descs]
        tpsa_values = [d["TPSA"] for d in linker_descs]
        aromatic_fraction_values = [d["aromatic_fraction"] for d in linker_descs]
        
        total_MolWt.append(sum(molwt_values))
        mean_MolWt.append(sum(molwt_values) / len(molwt_values))
        max_MolWt.append(max(molwt_values))
        
        total_TPSA.append(sum(tpsa_values))
        mean_TPSA.append(sum(tpsa_values) / len(tpsa_values))
        max_TPSA.append(max(tpsa_values))
        
        total_HBD.append(sum(d["HBD"] for d in linker_descs))
        total_HBA.append(sum(d["HBA"] for d in linker_descs))
        
        total_aromatic_atom_count.append(sum(d["aromatic_atom_count"] for d in linker_descs))
        mean_aromatic_fraction.append(sum(aromatic_fraction_values) / len(aromatic_fraction_values))
        
        total_heavy_atom_count_rdkit.append(sum(d["heavy_atom_count"] for d in linker_descs))
        total_ring_count.append(sum(d["ring_count"] for d in linker_descs))
        total_rotatable_bond_count.append(sum(d["rotatable_bond_count"] for d in linker_descs))
    
    # Use 0 if no valid linker is found
    else:
        total_MolWt.append(0)
        mean_MolWt.append(0)
        max_MolWt.append(0)
        
        total_TPSA.append(0)
        mean_TPSA.append(0)
        max_TPSA.append(0)
        
        total_HBD.append(0)
        total_HBA.append(0)
        total_aromatic_atom_count.append(0)
        mean_aromatic_fraction.append(0)
        total_heavy_atom_count_rdkit.append(0)
        total_ring_count.append(0)
        total_rotatable_bond_count.append(0)

In [76]:
# Add the new descriptor columns to the dataframe
df["total_MolWt"] = total_MolWt
df["mean_MolWt"] = mean_MolWt
df["max_MolWt"] = max_MolWt

df["total_TPSA"] = total_TPSA
df["mean_TPSA"] = mean_TPSA
df["max_TPSA"] = max_TPSA

df["total_HBD"] = total_HBD
df["total_HBA"] = total_HBA

df["total_aromatic_atom_count"] = total_aromatic_atom_count
df["mean_aromatic_fraction"] = mean_aromatic_fraction

df["total_heavy_atom_count_rdkit"] = total_heavy_atom_count_rdkit
df["total_ring_count"] = total_ring_count
df["total_rotatable_bond_count"] = total_rotatable_bond_count

In [ ]:
# List the physicochemical descriptor columns
physchem_cols = [
    "total_MolWt",
    "mean_MolWt",
    "max_MolWt",
    "total_TPSA",
    "mean_TPSA",
    "max_TPSA",
    "total_HBD",
    "total_HBA",
    "total_aromatic_atom_count",
    "mean_aromatic_fraction",
    "total_heavy_atom_count_rdkit",
    "total_ring_count",
    "total_rotatable_bond_count"
]

# Check the first few rows
df[physchem_cols].head()


,total_MolWt,mean_MolWt,max_MolWt,total_TPSA,mean_TPSA,max_TPSA,total_HBD,total_HBA,total_aromatic_atom_count,mean_aromatic_fraction,total_heavy_atom_count_rdkit,total_ring_count,total_rotatable_bond_count
0,164.116,164.116,164.116,80.26,80.260,80.26,0,4,6,0.500000,12,1,2
1,164.116,164.116,164.116,80.26,80.260,80.26,0,4,6,0.500000,12,1,2
2,218.086,218.086,218.086,80.26,80.260,80.26,0,4,6,0.400000,15,1,2
3,418.310,209.155,224.168,188.21,94.105,98.72,0,11,12,0.401786,30,2,7
4,360.318,180.159,248.278,160.52,80.260,80.26,0,8,6,0.166667,26,1,5


In [83]:
# Check the new df shape:
print(df.shape)
print()
# Check missing Values
print(df[physchem_cols].isna().sum())
print()
# Check summary statistics
print(df[physchem_cols].describe())



(25928, 56)

total_MolWt                     0
mean_MolWt                      0
max_MolWt                       0
total_TPSA                      0
mean_TPSA                       0
max_TPSA                        0
total_HBD                       0
total_HBA                       0
total_aromatic_atom_count       0
mean_aromatic_fraction          0
total_heavy_atom_count_rdkit    0
total_ring_count                0
total_rotatable_bond_count      0
dtype: int64

        total_MolWt    mean_MolWt     max_MolWt    total_TPSA     mean_TPSA  \
count  25928.000000  25928.000000  25928.000000  25928.000000  25928.000000   
mean     826.263195    290.151288    362.286264    263.737346     94.625414   
std      385.977325    114.318492    141.997062    116.386061     40.032751   
min       30.030000     30.030000     30.030000     24.720000     24.720000   
25%      558.400750    216.172000    274.236000    186.300000     69.366667   
50%      790.592000    275.500000    343.303000    240.78

In [86]:
# Check ring count for each linker separately and find the maximum single-linker ring count
max_single_linker_ring_count = 0
max_single_linker_info = None

for idx, row in df.iterrows():
    for col in linker_cols:
        desc = calculate_linker_descriptors(row[col])
        
        if desc is not None:
            if desc["ring_count"] > max_single_linker_ring_count:
                max_single_linker_ring_count = desc["ring_count"]
                max_single_linker_info = (idx, col, row[col])

print("Max single-linker ring count:", max_single_linker_ring_count)
print("Row index:", max_single_linker_info[0])
print("Linker column:", max_single_linker_info[1])
print("SMILES:", max_single_linker_info[2])

Max single-linker ring count: 40
Row index: 347
Linker column: linker_1
SMILES: N#Cc1c2ccc([C]([O-])[O-])c1[C]N1N3[C]c4c(C(=O)[O-])ccc(c4C#N)-c4ccc(C(=O)[O-])c(c4C#N)[C][N]N4[C]c5c6c7c8c(c5C(=O)[O-])[C]N=N[C]c5c9c([C]([O-])[O-])c%10c(c5-c5cc(c([C]([O-])[O-])c(c5)[C][N]3)[C][N][N][C]8)[C][N][N][C]c3cc5cc(c3C(=O)[O-])[C][N]N3[C]c8c(C(=O)[O-])ccc(c8C#N)-c8ccc(C(=O)[O-])c(c8C#N)[C][N]N([C]9)N([C]c8c(C(=O)[O-])ccc(c8C#N)-c8cc9c(C(=O)[O-])cc8[C]N=N[C]c8cc(c(cc8C(=O)[O-])[C]N=N[C]c8cc-2c(cc8[C]([O-])[O-])[C]N=N[C]9)-c2ccc(C(=O)[O-])c(c2C#N)[C]N34)N2[C]c3c([C]([O-])[O-])ccc(c3C#N)-c3ccc([C]([O-])[O-])c(c3C#N)[C][N]N1[C]c1c(c-5c(c(c1C(=O)[O-])[C]N=N[C]6)[C][N][N][C]c1cc-7cc(c1C(=O)[O-])[C][N]2)[C]N=N[C]%10


In [87]:
# Check MOFs with unusually high total ring count
df[df["total_ring_count"] > 20][
    ["filename", "total_ring_count", "total_MolWt", "total_heavy_atom_count_rdkit"]
].sort_values("total_ring_count", ascending=False)

,filename,total_ring_count,total_MolWt,total_heavy_atom_count_rdkit
23294,hMOF-4269,56,1841.441,135
30,hMOF-10029,43,1914.006,146
347,hMOF-10325,40,3212.346,246
414,hMOF-10390,31,1912.172,148
13808,hMOF-2448,28,7313.304,286
5516,hMOF-15399,28,2463.662,179
5737,hMOF-15601,27,2162.472,168
6578,hMOF-16381,26,1607.164,124
25127,hMOF-6229,26,2414.606,186
1629,hMOF-11601,25,1886.134,146


----

### Descriptor Validation Summary

The physicochemical descriptor features were successfully calculated for 25,928 MOFs.

No missing values were found in the new descriptor columns. The descriptor ranges are chemically reasonable:
- Mean aromatic fraction ranges from 0 to 1.
- Total heavy atom count ranges from 2 to 286.
- Total ring count ranges from 0 to 56.
- Total HBA ranges from 2 to 66.
- Total MolWt shows a wide range, reflecting variation in linker size and number of linkers.

These checks suggest that the descriptor calculation worked correctly and the features are ready for further validation and modeling.

----

In [94]:
# Save the extended descriptor dataset
from pathlib import Path

# Define output file path
output_path = Path("/home/susan/mof-co2-adsorption/data/processed/hmof_linker_extended_descriptors.csv")

# Create the folder if it does not already exist
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save dataframe as CSV
df.to_csv(output_path, index=False)

In [96]:
# Reload saved file to confirm it saved correctly
saved_df = pd.read_csv(output_path, low_memory=False)

print(saved_df.shape)

(25928, 56)


In [97]:
# Check constant / near-constant descriptor columns
df[physchem_cols].nunique().sort_values()

total_HBD                          25
total_ring_count                   33
total_HBA                          45
total_rotatable_bond_count         51
total_aromatic_atom_count          62
total_heavy_atom_count_rdkit      157
max_TPSA                          570
total_TPSA                       2397
mean_TPSA                        2481
max_MolWt                        3798
mean_aromatic_fraction           6108
mean_MolWt                      13678
total_MolWt                     13943
dtype: int64

In [100]:
# Check correlations between physicochemical descriptor columns
corr_matrix = df[physchem_cols].corr()

corr_matrix



,total_MolWt,mean_MolWt,max_MolWt,total_TPSA,mean_TPSA,max_TPSA,total_HBD,total_HBA,total_aromatic_atom_count,mean_aromatic_fraction,total_heavy_atom_count_rdkit,total_ring_count,total_rotatable_bond_count
total_MolWt,1.000000,0.848610,0.816978,0.338382,0.077142,0.082288,-0.010605,0.396439,0.784751,0.497550,0.862828,0.753619,0.413986
mean_MolWt,0.848610,1.000000,0.914667,0.214490,0.234604,0.181467,-0.032587,0.259258,0.680036,0.514434,0.707566,0.674610,0.346581
max_MolWt,0.816978,0.914667,1.000000,0.180173,0.152453,0.158046,-0.076577,0.225589,0.597163,0.394900,0.648127,0.607386,0.338879
total_TPSA,0.338382,0.214490,0.180173,1.000000,0.829269,0.841417,0.679355,0.957580,0.375483,0.148420,0.530883,0.360533,0.213588
mean_TPSA,0.077142,0.234604,0.152453,0.829269,1.000000,0.945596,0.643054,0.764150,0.165135,0.056427,0.262786,0.179369,0.081362
max_TPSA,0.082288,0.181467,0.158046,0.841417,0.945596,1.000000,0.657156,0.782501,0.176013,0.070425,0.271872,0.187910,0.078393
total_HBD,-0.010605,-0.032587,-0.076577,0.679355,0.643054,0.657156,1.000000,0.593267,0.132783,0.122483,0.101707,0.120300,-0.078176
total_HBA,0.396439,0.259258,0.225589,0.957580,0.764150,0.782501,0.593267,1.000000,0.403723,0.158003,0.590714,0.384701,0.368867
total_aromatic_atom_count,0.784751,0.680036,0.597163,0.375483,0.165135,0.176013,0.132783,0.403723,1.000000,0.799858,0.890242,0.929247,0.331768
mean_aromatic_fraction,0.497550,0.514434,0.394900,0.148420,0.056427,0.070425,0.122483,0.158003,0.799858,1.000000,0.551734,0.726046,0.034894


In [101]:
# Show strong correlations only
strong_corr = corr_matrix.abs().unstack().sort_values(ascending=False)

# Remove self-correlations
strong_corr = strong_corr[strong_corr < 1]

strong_corr.head(20)

total_HBA                     total_TPSA                      0.957580
total_TPSA                    total_HBA                       0.957580
mean_TPSA                     max_TPSA                        0.945596
max_TPSA                      mean_TPSA                       0.945596
total_aromatic_atom_count     total_ring_count                0.929247
total_ring_count              total_aromatic_atom_count       0.929247
mean_MolWt                    max_MolWt                       0.914667
max_MolWt                     mean_MolWt                      0.914667
total_heavy_atom_count_rdkit  total_aromatic_atom_count       0.890242
total_aromatic_atom_count     total_heavy_atom_count_rdkit    0.890242
total_MolWt                   total_heavy_atom_count_rdkit    0.862828
total_heavy_atom_count_rdkit  total_MolWt                     0.862828
total_ring_count              total_heavy_atom_count_rdkit    0.859007
total_heavy_atom_count_rdkit  total_ring_count                0.859007
mean_M

### Descriptor Validation Summary

The physicochemical descriptor columns were checked for missing values, constant features, and strong correlations.

No missing values were found in the new descriptor columns. The constant-feature check also showed that none of the descriptors were constant or near-empty. The number of unique values ranged from 25 for `total_HBD` to 13,943 for `total_MolWt`, indicating that all descriptor columns contain useful variation across the MOF dataset.

Correlation analysis showed that several descriptors are strongly correlated. This is expected because some features describe related chemical properties. For example, `total_HBA` was highly correlated with `total_TPSA`, since hydrogen-bond acceptor atoms contribute strongly to polar surface area. Similarly, `total_aromatic_atom_count`, `total_ring_count`, and `total_heavy_atom_count_rdkit` were strongly correlated because larger aromatic linkers tend to contain more rings and heavy atoms. The molecular-weight descriptors (`total_MolWt`, `mean_MolWt`, and `max_MolWt`) were also correlated, reflecting linker-size effects.

These correlated descriptors were retained for the first modeling comparison because tree-based models such as Random Forest and XGBoost can handle correlated input features. Later, model performance, feature importance, and SHAP analysis will be used to determine which chemistry descriptors add the most predictive value beyond the geometry-only features.